# Compute fitness effects for subset trees

Use expected rates from the global neutral model to compute fitness effects for each subset (host, geographic, temporal).

In [1]:
import pandas as pd
import numpy as np

## Read data

In [2]:
# Read subset counts
counts_df = pd.read_csv('../results/subset_counts.csv', keep_default_na=False, low_memory=False)
counts_df = counts_df.replace('', np.nan)
counts_df['codon_site'] = counts_df['codon_site'].astype(str)
print(f'Subset counts: {len(counts_df):,} rows')
print(f'Subsets: {sorted(counts_df["subset"].unique())}')

Subset counts: 3,626,625 rows
Subsets: ['asia', 'avian', 'early', 'europe', 'human', 'late', 'north_america']


In [3]:
# Read expected rates from the global neutral model
predicted_rates_df = pd.read_csv('../results/expected_rates.csv', keep_default_na=False)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200380
1,AC,HA,AAC,0.315456
2,AC,HA,AAG,0.246664
3,AC,HA,AAT,0.308056
4,AC,HA,CAA,0.178563


## Compute expected counts and filter

In [4]:
counts_cutoff = 10

actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    .query('actual_count >= @counts_cutoff | expected_count >= @counts_cutoff')
)
print(f'Rows after filtering (>= {counts_cutoff} actual or expected): {len(actual_expected_df):,}')
actual_expected_df.head()

Rows after filtering (>= 10 actual or expected): 248,282


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,subset,subset_type,evo_opp,rate,predicted_rate,expected_count
444,54,A54C,A,C,NP,3,18,GAA,GAC,E,...,all,NP,NP_all,1494,avian,host,21.056894,0.6648653803363107,0.338733,7.132663
726,2079,A2079C,A,C,PB2,3,693,GGA,GGC,G,...,all,PB2,PB2_all,2277,early,temporal,21.859464,0.45746775424920644,0.129274,2.825860
937,835,A835C,A,C,M2,3,49,AAA,AAC,K,...,all,MP,MP_all,979,late,temporal,33.745659,0.08890032387928685,0.429849,14.505545
966,2037,A2037C,A,C,PB2,3,679,CCA,CCC,P,...,all,PB2,PB2_all,2277,early,temporal,37.497585,0.26668384437000775,0.209385,7.851423
1049,6,A6C,A,C,PA,3,2,GAA,GAC,E,...,all,PA,PA_all,2148,human,host,56.006052,0.0,0.249623,13.980427


## Compute amino-acid-level fitness effects

In [5]:
pseudo_count = 0.5

groupby_cols = [
    'subset', 'subset_type', 'subtype', 'segment', 'gene',
    'codon_site', 'wt_aa', 'mut_aa', 'aa_mut', 'mut_class'
]

aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x:
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)

aa_fitness_df.to_csv('../results/subset_aa_fitness_effects.csv', index=False)
print(f'AA-level fitness effects: {len(aa_fitness_df):,} rows')
aa_fitness_df.head()

AA-level fitness effects: 173,234 rows


,subset,subset_type,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,asia,geographic,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,30.581079,-4.129746
1,asia,geographic,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,13.846674,-3.356665
2,asia,geographic,H1,HA,HA,10,Y,C,Y10C,nonsynonymous,26,23.617486,0.094208
3,asia,geographic,H1,HA,HA,10,Y,H,Y10H,nonsynonymous,21,26.990156,-0.245775
4,asia,geographic,H1,HA,HA,10,Y,Y,Y10Y,synonymous,81,29.269693,1.007112


In [6]:
# Summary by subset and mutation class
aa_fitness_df.groupby(['subset', 'mut_class']).size().unstack(fill_value=0)

mut_class,nonsense,nonsynonymous,synonymous
subset,,,
asia,845,16615,5963
avian,739,13535,4779
early,1300,20287,5905
europe,811,15232,5440
human,1639,23397,5403
late,1264,19234,5796
north_america,1076,18213,5761
